# 02 - Entrenamiento y evaluación
Entrena un Random Forest, muestra métricas y exporta el modelo y sus etiquetas.

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
dataset = pd.read_csv(RAIZ / 'data' / 'processed' / 'dataset.csv')
X = dataset.drop(columns='label')
codificador = LabelEncoder()
y = codificador.fit_transform(dataset['label'])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
modelo = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)
print(classification_report(y_test, predicciones, target_names=codificador.classes_))

In [ ]:
matriz = confusion_matrix(y_test, predicciones)
sns.heatmap(matriz, annot=True, fmt='d', xticklabels=codificador.classes_, yticklabels=codificador.classes_)
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.tight_layout()
plt.show()

In [ ]:
directorio = RAIZ / 'models'
directorio.mkdir(exist_ok=True)
joblib.dump(modelo, directorio / 'modelo_senas.pkl')
labels = {str(i): nombre for i, nombre in enumerate(codificador.classes_)}
(directorio / 'labels.json').write_text(
    json.dumps(labels, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Modelo y etiquetas exportados correctamente.')